# Lesson 31 — Dynamic Optimization and Optimal Control

## Learning objectives

1. State an optimal-control problem (states, controls, path and boundary constraints, Bolza objective) and explain the *simultaneous* (direct transcription) approach that turns it into one large, sparse NLP.
2. Discretize ODE/DAE models with finite differences and with orthogonal collocation on finite elements, and predict their order of accuracy.
3. Build dynamic models with `discopt.dae` (`ContinuousSet`, `DAEBuilder`, `add_state` / `add_control`, `set_ode`, `integral`, `state_at`) and solve optimal-control problems, including free-final-time problems by time scaling.
4. Fit kinetic parameters to trajectory data and quantify their uncertainty.
5. Separate the two errors of a transcribed solution (state discretization and control parametrization), and measure how the number of finite elements and collocation points trades accuracy against problem size.
6. Recognize mixed-integer dynamic optimization (switched controls) and why its integer solutions *chatter*.

*Prerequisites:* constrained NLP and KKT conditions (Lesson 9), interior-point methods (Lesson 12). Collocation is new.

## 1. The optimal-control problem

A dynamic optimization problem chooses a control trajectory $u(t)$ (and possibly parameters $p$) for a system described by differential-algebraic equations {cite:p}`Biegler2010,Betts2010`:

$$
\begin{aligned}
\min_{u(\cdot),\,p}\quad & \Phi\big(x(t_f)\big) + \int_{t_0}^{t_f} L\big(x(t), u(t), p\big)\,dt \\
\text{s.t.}\quad & \dot x(t) = f\big(x(t), z(t), u(t), p\big), \qquad x(t_0) = x_0, \\
& 0 = g\big(x(t), z(t), u(t), p\big), \\
& x^L \le x(t) \le x^U,\quad u^L \le u(t) \le u^U, \qquad \psi\big(x(t_f)\big) = 0 .
\end{aligned}
$$

Here $x$ are the **differential states**, $z$ the **algebraic states** (an index-1 DAE when $\partial g/\partial z$ is nonsingular), $\Phi$ the *Mayer* (terminal) term and $\int L$ the *Lagrange* (running) term; together they form a *Bolza* objective. The problem is infinite dimensional, and there are three families of methods for it:

| Approach | Idea | Typical issue |
|---|---|---|
| **Indirect** (Pontryagin) | Derive the adjoint ODEs and optimality conditions, then solve a boundary-value problem | Needs the switching structure in advance; hard with path constraints |
| **Sequential** (single shooting) | Parametrize only $u$; an ODE integrator computes $x(u)$ inside every NLP function evaluation | Unstable dynamics blow up; path constraints only at grid points |
| **Simultaneous** (direct transcription) | Discretize $x$, $z$ *and* $u$; the ODE becomes algebraic equality constraints of one large, sparse NLP | Large NLP, which needs a sparse interior-point solver |

`discopt.dae` implements the **simultaneous** approach {cite:p}`Cuthrell1987,Biegler2010`. The solver sees an ordinary NLP, so everything from the intermediate track applies: KKT conditions, the sparse interior-point method {cite:p}`Wachter2006`, and the same `m.solve()` call. The ODE is only satisfied *at the solution*, not at every iterate, and that is what makes the approach robust for unstable systems and natural for state bounds.

## 2. Discretization: finite differences and orthogonal collocation

Split $[t_0, t_f]$ into $N$ **finite elements** of width $h_i$. The simplest transcription is **backward (implicit) Euler**, a finite-difference scheme with one equation per grid point:

$$x_{i+1} = x_i + h\,f(x_{i+1}, u_i), \qquad \text{global error } O(h).$$

**Orthogonal collocation on finite elements** represents the state inside element $i$ by a polynomial of degree $K$ through the element start $t_{i,0}$ and $K$ *collocation points* $t_{i,j} = t_{i,0} + h_i\tau_j$, and enforces the ODE exactly at those points {cite:p}`Biegler2010`:

$$
\sum_{k=0}^{K} A_{jk}\, x_{i,k} \;=\; h_i\, f\big(x_{i,j}, z_{i,j}, u_i\big), \qquad j = 1,\dots,K,
$$

where $A_{jk} = \ell_k'(\tau_j)$ is the derivative of the $k$-th Lagrange basis polynomial (the *differentiation matrix*). Continuity of $x$ links neighbouring elements. With **Radau** points the last point is $\tau_K = 1$, so continuity is simply $x_{i+1,0} = x_{i,K}$, and the scheme is the implicit Radau IIA Runge–Kutta method. At element boundaries its error is $O(h^{2K-1})$: order 1 for $K=1$ (identical to backward Euler), 3 for $K=2$, and 5 for $K=3$. Gauss–**Legendre** points reach $O(h^{2K})$, but $\tau = 1$ is not a collocation point, so continuity needs an extrapolation. Radau is the default because it is stiffly accurate and handles DAEs well {cite:p}`Biegler2010`.

`discopt.dae.polynomials` exposes the building blocks:

In [ ]:
import time
import numpy as np
import matplotlib.pyplot as plt
import discopt.modeling as dm
from discopt.dae import (ContinuousSet, DAEBuilder, FDBuilder,
                         radau_roots, legendre_roots, collocation_matrix)

for K in (1, 2, 3):
    print(f"K={K}  Radau tau = {np.round(radau_roots(K), 4)}   Legendre tau = {np.round(legendre_roots(K), 4)}")

A, w = collocation_matrix(3, "radau")
print("\nRadau differentiation matrix A (K=3), shape", A.shape, "= K x (K+1):")
print(np.round(A, 3))
# Each row differentiates a polynomial exactly: rows sum to 0 (derivative of a constant).
print("row sums:", np.round(A.sum(axis=1), 12))

### Measuring the order of accuracy

Transcribe $\dot x = -x$, $x(0) = 1$ on $[0, 5]$ with no decision variables (a pure simulation: `m.minimize(0)`), and compare $x(5)$ with $e^{-5}$. `FDBuilder` uses the same interface as `DAEBuilder`, and its states live on the grid points only.

In [ ]:
def decay_error(nfe, kind):
    m = dm.Model("decay")
    if kind == "FD (backward Euler)":
        cs = ContinuousSet("t", bounds=(0, 5), nfe=nfe)
        b = FDBuilder(m, cs, method="backward")
    else:
        K = int(kind[-1])
        cs = ContinuousSet("t", bounds=(0, 5), nfe=nfe, ncp=K, scheme="radau")
        b = DAEBuilder(m, cs)
    b.add_state("x", initial=1.0, bounds=(0, 2))
    b.set_ode(lambda t, s, a, c: {"x": -s["x"]})
    b.discretize()
    m.minimize(0)
    r = m.solve()
    t, x = b.extract_solution(r, "x")
    return abs(x[-1] - np.exp(-5.0))

kinds = ["FD (backward Euler)", "Radau K=1", "Radau K=2", "Radau K=3"]
nfes = [5, 10, 20, 40]
err = {k: [decay_error(n, k) for n in nfes] for k in kinds}

print(f"{'nfe':>4} " + " ".join(f"{k:>20}" for k in kinds))
for i, n in enumerate(nfes):
    print(f"{n:>4} " + " ".join(f"{err[k][i]:>20.2e}" for k in kinds))
for k in kinds:
    p = np.polyfit(np.log(nfes), np.log(err[k]), 1)[0]
    print(f"observed order {k:>20}: {-p:.2f}")

plt.figure(figsize=(5, 3.5))
for k in kinds:
    plt.loglog(nfes, err[k], "o-", label=k)
plt.xlabel("finite elements N"); plt.ylabel("|x(5) - exp(-5)|")
plt.legend(fontsize=8); plt.grid(alpha=0.3); plt.tight_layout(); plt.show()

Backward Euler and one-point Radau give the *same* numbers (they are the same scheme), and each extra collocation point gains two orders. The cost is size: a Radau model has $N(K+1)$ values per state, versus $N + 1$ for finite differences. In practice, 3 Radau points on a moderate number of elements beat hundreds of Euler steps in both accuracy and NLP size.

## 3. Anatomy of a `discopt.dae` model

| Object / call | What it creates |
|---|---|
| `ContinuousSet("t", bounds=(t0, tf), nfe=N, ncp=K, scheme="radau")` | the time grid (optionally `element_boundaries=` for a non-uniform grid) |
| `dae = DAEBuilder(m, cs)` | a transcriber that adds variables and constraints to model `m` |
| `dae.add_state(name, initial=, bounds=)` | differential state, variable of shape `(N, K+1)`; `initial` fixes `x[0, 0]` |
| `dae.add_algebraic(name, bounds=)` | algebraic state $z$, shape `(N, K)` (collocation points only) |
| `dae.add_control(name, bounds=)` | **piecewise-constant** control, shape `(N,)` |
| `dae.set_ode(rhs)`, `dae.set_algebraic(res)` | `rhs(t, states, algebraics, controls) -> {state: derivative}` |
| `dae.discretize()` | emits collocation and continuity constraints |
| `dae.get_state(name)` | the underlying `discopt` variable (states, algebraics *and* controls) |
| `dae.integral(L)` | Radau/Gauss quadrature of $\int L\,dt$ as an expression |
| `dae.state_at(name, t)` | the collocation polynomial evaluated at any time $t$, as an expression |
| `dae.extract_solution(r, name)` | `(t, x)` arrays with duplicate element boundaries removed |

Two details matter in practice:

* `set_ode`'s callable is called **once** with vector expressions of shape `(N, K)` for all collocation points at the same time, so write it with `discopt` operators (`dm.exp`, `**`, `*`), not `numpy` functions and not `if` statements on values. The terminal state is `x[-1, -1]`.
* The transcribed problem is a nonconvex NLP whenever $f$ is nonlinear. A default `m.solve()` then starts spatial branch-and-bound, which cannot bound the thousands of nonlinear equality constraints of a transcription and runs to its time limit. For dynamic optimization, ask for a **local** solve with `m.solve(skip_convex_check=True)`. `discopt` then honestly reports `status == "feasible"` (a KKT point, not a certified global optimum), `r.gap` is `None`, and a log line *"No valid dual bound was produced…"* says the same thing. That message is expected for every local dynamic solve below.

## 4. Optimal temperature profile for a batch reactor

The classic example of the simultaneous approach {cite:p}`Biegler2010`: series reactions $A \to B \to C$ in a batch reactor, second order in $A$ and first order in $B$, with temperature $T(t) \in [298, 398]$ K as the control. The goal is the most intermediate $B$ at $t_f = 1$ h:

$$
\max_{T(\cdot)}\; b(1) \quad \text{s.t.}\quad
\dot a = -k_1(T)\,a^2,\quad \dot b = k_1(T)\,a^2 - k_2(T)\,b,\quad a(0)=1,\ b(0)=0,
$$

with $k_1 = 4000\,e^{-2500/T}$ and $k_2 = 6.2\times10^5\,e^{-5000/T}$. Because $E_2 > E_1$, high temperature favours the unwanted second reaction. The trade-off is to run hot early, while $a$ is large, and to cool as $B$ accumulates.

In [ ]:
def batch_reactor(nfe=20, ncp=3, constant_T=False):
    m = dm.Model("batch_reactor")
    cs = ContinuousSet("t", bounds=(0.0, 1.0), nfe=nfe, ncp=ncp, scheme="radau")
    dae = DAEBuilder(m, cs)
    dae.add_state("a", initial=1.0, bounds=(0, 1))
    dae.add_state("b", initial=0.0, bounds=(0, 1))
    dae.add_control("T", bounds=(298, 398))

    def rhs(t, s, alg, c):
        k1 = 4000.0 * dm.exp(-2500.0 / c["T"])
        k2 = 6.2e5 * dm.exp(-5000.0 / c["T"])
        return {"a": -k1 * s["a"] ** 2, "b": k1 * s["a"] ** 2 - k2 * s["b"]}

    dae.set_ode(rhs)
    dae.discretize()
    if constant_T:                       # an isothermal reactor: one temperature for all elements
        T = dae.get_state("T")
        m.subject_to(T[1:] == T[:-1])
    m.maximize(dae.get_state("b")[-1, -1])
    return m, dae

m, dae = batch_reactor(nfe=20, ncp=3)
t0 = time.perf_counter()
r = m.solve(skip_convex_check=True)        # local NLP solve (see Section 3)
print(f"status={r.status}  b(1) = {r.objective:.5f}  ({time.perf_counter() - t0:.2f} s)")

mi, daei = batch_reactor(nfe=20, ncp=3, constant_T=True)
ri = mi.solve(skip_convex_check=True)
print(f"best isothermal: T = {ri.value(daei.get_state('T'))[0]:.1f} K, b(1) = {ri.objective:.5f}")

T_opt = r.value(dae.get_state("T"))
ta, a_opt = dae.extract_solution(r, "a")
tb, b_opt = dae.extract_solution(r, "b")
edges = np.linspace(0, 1, len(T_opt) + 1)
fig, ax = plt.subplots(1, 2, figsize=(9, 3.2))
ax[0].stairs(T_opt, edges); ax[0].set_xlabel("t [h]"); ax[0].set_ylabel("T [K]")
ax[0].set_title("optimal temperature (piecewise constant)")
ax[1].plot(ta, a_opt, label="a"); ax[1].plot(tb, b_opt, label="b")
ax[1].set_xlabel("t [h]"); ax[1].legend(); ax[1].set_title("concentrations")
plt.tight_layout(); plt.show()

The optimal policy starts near 372 K and cools steadily to about 327 K. It gives $b(1) \approx 0.6105$, against $0.6059$ for the best single temperature (335 K). The improvement is small but typical: the value of dynamic optimization often lies in operating closer to constraints or reducing by-products.

**Verify by simulation.** A transcription is only an approximation of the ODE. The standard check is to take the optimal control, integrate the true ODE with a high-accuracy integrator, and compare. We will reuse this check in Section 7.

In [ ]:
import logging
from scipy.integrate import solve_ivp
# Keep solver logging quiet for the rest of the lesson.
logging.getLogger("discopt.solver").setLevel(logging.ERROR)

def simulate_batch(T_profile, tf=1.0):
    # Integrate the reactor ODE element by element with the piecewise-constant T.
    y = np.array([1.0, 0.0])
    edges = np.linspace(0, tf, len(T_profile) + 1)
    for i, T in enumerate(T_profile):
        k1, k2 = 4000 * np.exp(-2500 / T), 6.2e5 * np.exp(-5000 / T)
        sol = solve_ivp(lambda t, y: [-k1 * y[0] ** 2, k1 * y[0] ** 2 - k2 * y[1]],
                        (edges[i], edges[i + 1]), y, method="Radau", rtol=1e-11, atol=1e-13)
        y = sol.y[:, -1]
    return y

b_sim = simulate_batch(T_opt)[1]
print(f"NLP b(1) = {r.objective:.8f}   simulated b(1) = {b_sim:.8f}   |diff| = {abs(r.objective - b_sim):.1e}")

## 5. Integral objectives, terminal constraints, and a certified optimum

Move a unit mass ($\ddot x = u$) from rest at $x=0$ to rest at $x=1$ in one second with minimum control energy $\int_0^1 u^2\,dt$. The dynamics are linear and the objective is a convex quadratic, so after transcription the NLP is **convex**. `discopt` proves this and certifies the *global* optimum with a single interior-point solve (`convex_fast_path`), so no `skip_convex_check` is needed. The continuous-time answer is $u^*(t) = 6 - 12t$ with $J^* = 12$. A piecewise-constant control on $N$ elements can only approach it from above.

In [ ]:
def min_energy(nfe):
    m = dm.Model("min_energy")
    cs = ContinuousSet("t", bounds=(0, 1), nfe=nfe, ncp=3)
    dae = DAEBuilder(m, cs)
    dae.add_state("x", initial=0.0, bounds=(-5, 5))
    dae.add_state("v", initial=0.0, bounds=(-5, 5))
    dae.add_control("u", bounds=(-20, 20))
    dae.set_ode(lambda t, s, a, c: {"x": s["v"], "v": c["u"]})
    dae.discretize()
    x, v = dae.get_state("x"), dae.get_state("v")
    m.subject_to(x[-1, -1] == 1.0)                     # terminal (boundary) conditions
    m.subject_to(v[-1, -1] == 0.0)
    m.minimize(dae.integral(lambda t, s, a, c: c["u"] ** 2))   # Lagrange term by quadrature
    return m, dae

for nfe in (5, 10, 20):
    m, dae = min_energy(nfe)
    r = m.solve()
    print(f"N={nfe:>3}: status={r.status:8s} convex_fast_path={r.convex_fast_path}  J={r.objective:.5f}")
u = r.value(dae.get_state("u"))
print("u on the first/last elements:", np.round(u[:2], 3), np.round(u[-2:], 3), " (exact 6 - 12t)")

**Free final time.** Minimum-time problems have an unknown horizon $t_f$. The standard trick is *time scaling*: introduce $\tau = t/t_f \in [0, 1]$, make $t_f$ a decision variable, and multiply every right-hand side by it, since $dx/d\tau = t_f\,f(x,u)$. In `discopt` that means `tf = m.continuous("tf", lb=..., ub=...)` captured in the `rhs` closure, with `m.minimize(tf)`. Exercise 2 applies this to the minimum-time version of the problem above.

## 6. Parameter estimation from trajectory data

Dynamic models usually contain uncertain kinetic parameters. Given measurements $y_{\ell}$ at times $t_\ell$, the estimation problem is itself a dynamic optimization {cite:p}`Bard1998,Biegler2010`:

$$\min_{\theta}\; \sum_\ell \big(x(t_\ell;\theta) - y_\ell\big)^2 \quad \text{s.t. the transcribed ODE.}$$

In the simultaneous approach, the parameters are ordinary `m.continuous` variables captured by the `rhs`. `dae.least_squares(state, t_data, y_data)` evaluates the collocation polynomial *at the measurement times* through `state_at`, so the data do not have to lie on the grid. We generate noisy data for $A \xrightarrow{k_1} B \xrightarrow{k_2} C$ with $k_1 = 2$, $k_2 = 0.7$, $\sigma = 0.01$.

In [ ]:
k1_true, k2_true, sigma = 2.0, 0.7, 0.01
def abc_rhs_np(k1, k2):
    return lambda t, y: [-k1 * y[0], k1 * y[0] - k2 * y[1]]
def simulate_abc(k1, k2, t_eval, y0=(1.0, 0.0)):
    sol = solve_ivp(abc_rhs_np(k1, k2), (0, t_eval[-1]), y0, t_eval=t_eval, rtol=1e-10, atol=1e-12)
    return sol.y

rng = np.random.default_rng(0)
t_data = np.linspace(0.25, 4.0, 16)
a_data, b_data = simulate_abc(k1_true, k2_true, t_data) + sigma * rng.standard_normal((2, t_data.size))

m = dm.Model("abc_fit")
k1 = m.continuous("k1", lb=0.01, ub=10.0)
k2 = m.continuous("k2", lb=0.01, ub=10.0)
cs = ContinuousSet("t", bounds=(0, 4), nfe=16, ncp=3)
dae = DAEBuilder(m, cs)
dae.add_state("a", initial=1.0, bounds=(0, 1.5))
dae.add_state("b", initial=0.0, bounds=(0, 1.5))
dae.set_ode(lambda t, s, al, c: {"a": -k1 * s["a"], "b": k1 * s["a"] - k2 * s["b"]})
dae.discretize()
m.minimize(dae.least_squares("a", t_data, a_data) + dae.least_squares("b", t_data, b_data))
r = m.solve(skip_convex_check=True)
k1_hat, k2_hat, ssr = float(r.value(k1)), float(r.value(k2)), r.objective
print(f"status={r.status}  k1={k1_hat:.4f}  k2={k2_hat:.4f}  SSR={ssr:.3e}  (truth 2.0, 0.7)")

**Uncertainty.** The Gauss–Newton covariance is $\mathrm{Cov}(\hat\theta) \approx s^2 (J^\top J)^{-1}$, where $s^2 = \mathrm{SSR}/(n - p)$ and $J_{\ell r} = \partial x(t_\ell)/\partial\theta_r$ is the **sensitivity** of the *trajectory* to the parameters {cite:p}`Bard1998`. Here $J$ must be the *total* derivative: the states move when $\theta$ moves, because they are tied to it through the ODE (the collocation equality constraints). We compute it by central finite differences of an accurate simulation.

`discopt.estimate.estimate_parameters` (with an `Experiment` subclass) automates point estimates, the FIM and confidence intervals. On a dynamic model it computes exactly this total derivative, by the implicit-function theorem on the collocation equalities: $dz/d\theta = -(\partial g/\partial z)^{-1}\,\partial g/\partial\theta$. (It refuses, rather than guessing, when an inequality is active at the solution, because the equalities alone then no longer determine $dz/d\theta$.) We first compute $J$ independently by finite differences of an accurate simulation, then cross-check with `estimate_parameters`.

In [ ]:
def sensitivity_jacobian(k1, k2, t_eval, y0=(1.0, 0.0), h=1e-6):
    cols = []
    for dk in [(h, 0.0), (0.0, h)]:
        up = simulate_abc(k1 + dk[0], k2 + dk[1], t_eval, y0)
        dn = simulate_abc(k1 - dk[0], k2 - dk[1], t_eval, y0)
        cols.append(((up - dn) / (2 * h)).ravel())    # stacks [a(t_l); b(t_l)]
    return np.column_stack(cols)

J = sensitivity_jacobian(k1_hat, k2_hat, t_data)
n_obs, n_par = J.shape[0], 2
s2 = ssr / (n_obs - n_par)
cov = s2 * np.linalg.inv(J.T @ J)
se = np.sqrt(np.diag(cov))
corr = cov[0, 1] / (se[0] * se[1])
print(f"k1 = {k1_hat:.4f} ± {se[0]:.4f}   k2 = {k2_hat:.4f} ± {se[1]:.4f}   corr = {corr:.2f}")
print(f"s = {np.sqrt(s2):.4f}  (true sigma = {sigma})")

In [ ]:
from discopt.estimate import Experiment, ExperimentModel, estimate_parameters

class ABCExperiment(Experiment):
    def create_model(self, **kwargs):
        m = dm.Model("abc_est")
        k1 = m.continuous("k1", lb=0.01, ub=10.0)
        k2 = m.continuous("k2", lb=0.01, ub=10.0)
        dae = DAEBuilder(m, ContinuousSet("t", bounds=(0, 4), nfe=16, ncp=3))
        dae.add_state("a", initial=1.0, bounds=(0, 1.5))
        dae.add_state("b", initial=0.0, bounds=(0, 1.5))
        dae.set_ode(lambda t, s, al, c: {"a": -k1 * s["a"], "b": k1 * s["a"] - k2 * s["b"]})
        dae.discretize()
        responses = {}                 # one scalar response per measured value
        for i, ti in enumerate(t_data):
            responses[f"a{i}"] = dae.state_at("a", float(ti))
            responses[f"b{i}"] = dae.state_at("b", float(ti))
        return ExperimentModel(model=m, unknown_parameters={"k1": k1, "k2": k2},
                               design_inputs={}, responses=responses,
                               measurement_error={key: sigma for key in responses})

data = {**{f"a{i}": a_data[i] for i in range(t_data.size)},
        **{f"b{i}": b_data[i] for i in range(t_data.size)}}
est = estimate_parameters(ABCExperiment(), data, initial_guess={"k1": 1.0, "k2": 1.0})
se_est = est.standard_errors
print(est.summary())
# estimate_parameters uses the declared sigma; the FD computation above used s.
print("FD standard errors rescaled by sigma/s:",
      np.round(se * sigma / np.sqrt(s2), 5), " vs", np.round([se_est["k1"], se_est["k2"]], 5))


Both estimates lie within about two standard errors of the truth, and the residual standard deviation recovers $\sigma$. The two uncertainty computations agree once they use the same noise level: `estimate_parameters` weights by the declared $\sigma$, while the finite-difference version estimated $s$ from the residuals. For several experiments with different initial conditions or inputs that share the same parameters, `discopt.dae.fit_trajectories` builds one collocation block per experiment on a shared model and returns a joint `least_squares()` objective (Exercise 4).

## 7. How fine should the discretization be?

A transcribed optimal-control solution carries **two** different errors:

1. **State discretization error.** The NLP's states only approximate the ODE solution for the chosen control. It is measured by $|b_{\text{NLP}}(t_f) - b_{\text{sim}}(t_f)|$, where the simulation integrates the true ODE under the optimal control, and it shrinks like $h^{2K-1}$.
2. **Control parametrization error.** A piecewise-constant $u$ on $N$ elements cannot represent the true continuous optimum. The simulated objective approaches the infinite-dimensional optimum (≈ 0.6106 here {cite:p}`Biegler2010`) only as $N \to \infty$, whatever $K$ is.

Problem size grows like $N(K+1)$ per state. Solve time also grows, but stays small for these sparse NLPs.

In [ ]:
print(f"{'K':>2} {'N':>4} {'#vars':>6} {'b NLP':>10} {'b sim':>10} {'|NLP-sim|':>10} {'time[s]':>8}")
for K in (1, 3):
    for N in (5, 10, 20, 40):
        m, dae = batch_reactor(nfe=N, ncp=K)
        t0 = time.perf_counter()
        r = m.solve(skip_convex_check=True)
        dt = time.perf_counter() - t0
        b_sim = simulate_batch(r.value(dae.get_state("T")))[1]
        print(f"{K:>2} {N:>4} {m.num_variables:>6} {r.objective:>10.6f} {b_sim:>10.6f} "
              f"{abs(r.objective - b_sim):>10.1e} {dt:>8.2f}")

With $K = 1$ (implicit Euler) the NLP *under-predicts* $b(t_f)$ by 1–8 % even at $N = 40$, so the optimizer is working on the wrong model. With $K = 3$ the state error is at the level of $10^{-6}$ to $10^{-9}$, and the remaining improvement with $N$ comes from the finer control. A sound workflow is to choose $K = 3$ Radau, increase $N$ until the simulated objective stops changing at the precision you care about, and always validate by simulation.

## 8. Mixed-integer dynamic optimization: switched controls

Some controls are discrete: a heater or valve that is on or off, a pump that is running or idle. Constraining a control to a binary per element, `y = m.binary(...)` with `u == y`, gives a **mixed-integer dynamic optimization** problem. When the dynamics are linear and the objective is convex, the result is a convex MIQP that `discopt`'s branch-and-bound solves to a certified optimum.

The relaxed problem ($0 \le u \le 1$) gives a lower bound. Typically it holds $u$ at a fractional value such as 0.5, and the integer solution imitates that value by *chattering*, switching on and off every element. Real actuators cannot switch arbitrarily often, so a limit on the number of switches is a natural extra constraint, and a costly one.

In [ ]:
def heater(nfe=10, binary=True, max_switches=None):
    # dx/dt = -x + 2u, x(0)=0; keep x near 1 on [0, 4] with an on/off heater u.
    m = dm.Model("heater")
    cs = ContinuousSet("t", bounds=(0, 4), nfe=nfe, ncp=2)
    dae = DAEBuilder(m, cs)
    dae.add_state("x", initial=0.0, bounds=(-1, 3))
    dae.add_control("u", bounds=(0, 1))
    dae.set_ode(lambda t, s, a, c: {"x": -s["x"] + 2 * c["u"]})
    dae.discretize()
    u = dae.get_state("u")
    if binary:
        y = m.binary("y", shape=(nfe,))
        m.subject_to(u == y)
        if max_switches is not None:
            sw = m.continuous("sw", shape=(nfe - 1,), lb=0, ub=1)   # sw_i >= |y_{i+1} - y_i|
            m.subject_to(sw >= y[1:] - y[:-1])
            m.subject_to(sw >= y[:-1] - y[1:])
            m.subject_to(dm.sum(sw) <= max_switches)
    m.minimize(dae.integral(lambda t, s, a, c: (s["x"] - 1.0) ** 2))
    return m, dae

for label, kw in [("relaxed 0<=u<=1", dict(binary=False)),
                  ("binary, <= 2 switches", dict(max_switches=2))]:
    m, dae = heater(**kw)
    t0 = time.perf_counter()
    r = m.solve(time_limit=60)
    print(f"{label:24s} status={r.status:8s} J={r.objective:.4f}  ({time.perf_counter() - t0:.1f} s)")
    print("   u =", np.round(r.value(dae.get_state("u")), 2) + 0.0)

The relaxation settles at $u \approx 0.5$ (holding $x = 1$ needs $2u = x$). With only two switches allowed, the heater must run in long blocks, and the tracking cost roughly triples. Exercise 6 maps out the trade-off between the switch limit and the cost.

## Summary

* Direct transcription turns an optimal-control problem into one sparse NLP. `discopt.dae` builds it for you, and `m.solve(skip_convex_check=True)` finds a local optimum.
* Radau collocation with $K$ points has error $O(h^{2K-1})$. $K = 3$ on a modest grid is the usual default, and finite differences need far more points for the same accuracy.
* Validate every dynamic optimum by simulating the optimal control, and refine $N$ until the *simulated* objective converges.
* Parameter estimation is dynamic optimization with a least-squares objective. Its uncertainty needs the *total* sensitivity of the trajectory.
* Binary controls give mixed-integer dynamic optimization problems. Their relaxations chatter, and switching limits restore physical meaning at a price.

## References
{cite:p}`Biegler2010` is the main text for the simultaneous approach and collocation; {cite:p}`Betts2010` covers direct transcription in aerospace practice; {cite:p}`Cuthrell1987` is the original collocation-based DAE optimization paper; {cite:p}`Bard1998` for nonlinear parameter estimation; {cite:p}`Wachter2006` for the interior-point solver that makes large transcriptions tractable.